# [1-3] final-submission-v3

**差分メモ（2026-03-22）**

- 対象: `[1-3]final-submission-v3.ipynb`
- 元: `[1]final-submission-v3.ipynb` を最終提出向けに整理した系統
- 参照: 前処理/後処理は `notebooks/002/[4-8]submit-notebook-v1.ipynb` と同一ロジック
- モデル: ByT5-large / ByT5-base / ByT5-small の3モデル
- 主要設定: `max_input_length=1024`, `max_new_tokens=1024`（維持）
- デコード: 各モデル beam=4候補 + sample=2候補（計最大18候補）
- MBR: `CHRF(word_order=2)` を使用（chrF++相当）
- 安全策: OOM時に `[OOM]` ログを出しつつバッチ分割で継続、空出力は `<gap>` に統一、提出前 `validate_submission` を実施
- (インフラ)P100でも動くように、P100で実行が完了する公開ノートブックをコピーしてそこに実装

## Model paths
- ByT5-large: `/kaggle/input/datasets/tatsuokoshida/dpc-byt5-large-v002-6`
- ByT5-base : `/kaggle/input/datasets/tatsuokoshida/dpc-byt5-base-v002-6`
- ByT5-small: `/kaggle/input/notebooks/tatsuokoshida/2-8-dpc-starter-train-v1/byt5-akkadian-model/fulltrain_byt5-small_multitask`


In [1]:
#!/usr/bin/env python3
import os
import gc
import math
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import List
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'true')

'true'

In [2]:
def _cuda_bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        return bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
    except Exception:
        return False


@dataclass
class CFG:
    # Competition data path (Kaggle may mount it in either location)
    input_dirs: tuple[str, ...] = (
        '/kaggle/input/competitions/deep-past-initiative-machine-translation',
        '/kaggle/input/deep-past-initiative-machine-translation',
    )

    # Models
    byt5_large_dir: str = '/kaggle/input/datasets/tatsuokoshida/dpc-byt5-large-v002-6-15epochs'
    byt5_base_dir: str  = '/kaggle/input/datasets/tatsuokoshida/dpc-byt5-base-v002-6-15epochs'
    byt5_small_dir: str = '/kaggle/input/notebooks/tatsuokoshida/2-8-dpc-starter-train-v1/byt5-akkadian-model/fulltrain_byt5-small_multitask'

    output_dir: str = '/kaggle/working'

    # Tokenization / generation
    max_input_length: int = 1024
    max_new_tokens: int = 1024

    # Decoding
    num_beams: int = 8
    num_beam_cands: int = 4
    num_sample_cands: int = 2
    sample_top_p: float = 0.92
    sample_temperature: float = 0.75

    # Inference
    batch_size: int = 4
    num_workers: int = 0
    num_buckets: int = 6
    early_stopping: bool = True
    length_penalty: float = 1.3
    repetition_penalty: float = 1.2

    use_mixed_precision: bool = True
    use_better_transformer: bool = True
    use_bucket_batching: bool = True
    use_adaptive_beams: bool = True

    def __post_init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)

        if not torch.cuda.is_available():
            self.use_mixed_precision = False
            self.use_better_transformer = False

        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == 'cuda'
            and _cuda_bf16_supported()
        )

        assert self.num_beams >= self.num_beam_cands


cfg = CFG()
print('PyTorch:', torch.__version__)
print('Device :', cfg.device)
if cfg.device.type == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))
    print('BF16   :', _cuda_bf16_supported())
print('BF16 AMP:', cfg.use_bf16_amp)


PyTorch: 2.8.0+cu126
Device : cuda
GPU    : Tesla P100-PCIE-16GB
BF16   : True
BF16 AMP: True


In [3]:
# DIFF (2026-03-21): 前処理/後処理は `[2-8]dpc-starter-train-v1.ipynb` と同一ロジック前提（submit 側は build_inputs を利用）
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
from typing import List


PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

# --- Pre/Post processing (aligned to notebooks/006/lb-35-9-ensembling-post-processing-baseline.ipynb) ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# Unicode fraction map (all single-character Unicode vulgar fractions)
_UNICODE_FRAC_MAP = {
    (1, 2): "½", (1, 3): "⅓", (2, 3): "⅔",
    (1, 4): "¼", (3, 4): "¾",
    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
    (1, 6): "⅙", (5, 6): "⅚",
    (1, 7): "⅐",
    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞",
    (1, 9): "⅑",
    (1, 10): "⅒",
}
def _trunc_float(x: float, places: int = 4) -> float:
    factor = 10 ** places
    if x >= 0:
        return math.floor(x * factor) / factor
    return -math.floor(-x * factor) / factor

def _fmt_trunc(x: float, places: int = 4) -> str:
    return f"{_trunc_float(x, places):.{places}f}".rstrip("0").rstrip(".")

# Map the 4-decimal *truncated* representation to a Unicode fraction.
# (Host example: 1.333330000... -> 1.3333; 0.1666 -> ⅙)
_FRAC_DECIMAL_MAP_4 = {}
for (n, d), ch in _UNICODE_FRAC_MAP.items():
    _FRAC_DECIMAL_MAP_4[_fmt_trunc(n / d, 4)] = ch

# Decimals (incl. long floats)
_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d+)(?![\w/])")
_LONG_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d{5,})(?![\w/])")

# General fractions like "1/2" or mixed fractions like "2 1/2".
_MIXED_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s+(\d+)\s*/\s*(\d+)(?![\w/])")
_SIMPLE_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s*/\s*(\d+)(?![\w/])")

def _mixed_frac_repl(m: re.Match) -> str:
    ip = int(m.group(1))
    num = int(m.group(2))
    den = int(m.group(3))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return f"{ip} {ch}" if ip != 0 else ch
    return m.group(0)

def _simple_frac_repl(m: re.Match) -> str:
    num = int(m.group(1))
    den = int(m.group(2))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return ch
    return m.group(0)

def _canon_decimal_str(s: str) -> str:
    # Keep x.5 (e.g., 2.5) as-is by request.
    if re.fullmatch(r"[1-9]\d*\.5", s):
        return s

    m = re.fullmatch(r"(\d+)\.(\d+)", s)
    if not m:
        return s

    ip = int(m.group(1))
    frac_digits = m.group(2)

    # Truncate fractional part to 4 digits (no rounding), pad with zeros for lookup.
    frac4 = (frac_digits + "0000")[:4]
    frac_key = ("0." + frac4).rstrip("0").rstrip(".")
    ch = _FRAC_DECIMAL_MAP_4.get(frac_key)
    if ch and frac_key != "0":
        if ip == 0:
            return ch
        return f"{ip} {ch}"

    # Long-float shortening: truncate to 4 digits after the decimal.
    if len(frac_digits) > 4:
        return f"{ip}.{frac4}".rstrip("0").rstrip(".")

    return s


_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE    = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP   = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE   = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE  = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE       = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE       = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE         = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)

        # Uppercase determinatives are unwrapped, lowercase ones are converted to braces.
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        ser = ser.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        ser = ser.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_ALT_SLASH_PAIR_RE = re.compile(r"\b([^\W\d_]+)\s*/\s*([^\W\d_]+)\b")
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)

        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)

        s = s.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        s = s.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        s = s.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)

        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        # Keep the left alternative (e.g., "you / she" -> "you")
        s = s.str.replace(_ALT_SLASH_PAIR_RE, r"\1", regex=True)

        # Remove only curly quotes; keep straight quotes and apostrophes.
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)

        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

        return s.tolist()

_preprocessor = OptimizedPreprocessor()
_postprocessor = VectorizedPostprocessor()

def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Competition input directory not found. Tried: " + ", ".join(input_dirs)
    )


def build_inputs(test_df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]:
    if "id" not in test_df.columns:
        raise ValueError(f"test.csv must contain 'id'. columns={list(test_df.columns)}")
    ids = test_df["id"].astype(str).tolist()

    if "task" in test_df.columns:
        tasks = test_df["task"].astype(str).tolist()
    else:
        tasks = ["akk2en"] * len(test_df)

    if "source" in test_df.columns:
        sources = test_df["source"].astype(str).tolist()
    elif "transliteration" in test_df.columns:
        sources = test_df["transliteration"].astype(str).tolist()
    else:
        raise ValueError(
            "test.csv must contain either 'source' or 'transliteration'. "
            f"columns={list(test_df.columns)}"
        )

    inputs: list[str] = [""] * len(sources)

    idx_akk2en = [i for i, t in enumerate(tasks) if t == "akk2en"]
    idx_en2akk = [i for i, t in enumerate(tasks) if t == "en2akk"]

    src_akk = [sources[i] for i in idx_akk2en]
    src_en = [sources[i] for i in idx_en2akk]

    src_akk_pp = _preprocessor.preprocess_batch(src_akk) if src_akk else []
    src_en_pp = _postprocessor.postprocess_batch(src_en) if src_en else []

    for i, s_norm in zip(idx_akk2en, src_akk_pp):
        inputs[i] = PREFIX_AKK_EN + s_norm
    for i, s_norm in zip(idx_en2akk, src_en_pp):
        inputs[i] = PREFIX_EN_AKK + s_norm

    # Keep strict: unknown tasks are a hard error.
    unknown = sorted(set(tasks) - {"akk2en", "en2akk"})
    if unknown:
        raise ValueError(f"Unknown task(s): {unknown}")

    return ids, tasks, inputs


In [4]:
def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError('Competition input directory not found. Tried: ' + ', '.join(input_dirs))

def resolve_model_dir(model_dir: str) -> str:
    # model_dir は Kaggle Dataset/Notebook/Working のいずれでも良い
    candidates = [model_dir]
    for d in candidates:
        if Path(d).exists():
            return d
    raise FileNotFoundError(f'Model directory not found: {model_dir}')

input_dir = resolve_input_dir(cfg.input_dirs)
test_path = f'{input_dir}/test.csv'
print('Test:', test_path)
test_df = pd.read_csv(test_path)
print('Test rows:', len(test_df))
print('Columns:', list(test_df.columns))

ids, tasks, inputs = build_inputs(test_df)
print('Prepared inputs:', len(inputs))


Test: /kaggle/input/deep-past-initiative-machine-translation/test.csv
Test rows: 4
Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Prepared inputs: 4


In [5]:
# ==========================================
# 5. Multi-model candidate generation + MBR
# ==========================================
class TextDataset(Dataset):
    def __init__(self, texts: list[str], tasks: list[str]):
        if len(texts) != len(tasks):
            raise ValueError(f'len(texts)={len(texts)} != len(tasks)={len(tasks)}')
        self.texts = texts
        self.tasks = tasks

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        return {'index': idx, 'text': self.texts[idx], 'task': self.tasks[idx]}


class BucketBatchSampler(Sampler[list[int]]):
    def __init__(self, texts: list[str], batch_size: int, num_buckets: int):
        self.batch_size = batch_size
        lengths = [max(1, len(t.split())) for t in texts]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])

        bucket_size = max(1, len(sorted_idx) // max(1, num_buckets))
        self.buckets = [
            sorted_idx[i * bucket_size: None if i == num_buckets - 1 else (i + 1) * bucket_size]
            for i in range(num_buckets)
        ]

    def __iter__(self):
        for bucket in self.buckets:
            for i in range(0, len(bucket), self.batch_size):
                yield bucket[i:i + self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(bucket) / self.batch_size) for bucket in self.buckets)


def make_dataloader(tokenizer, texts: list[str], tasks: list[str]) -> DataLoader:
    def collate_fn(batch: list[dict]):
        b_indices = [x['index'] for x in batch]
        b_texts = [x['text'] for x in batch]
        b_tasks = [x['task'] for x in batch]
        enc = tokenizer(
            b_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=cfg.max_input_length,
        )
        enc['indices'] = b_indices
        enc['tasks'] = b_tasks
        return enc

    ds = TextDataset(texts, tasks)

    if cfg.use_bucket_batching:
        sampler = BucketBatchSampler(texts, batch_size=cfg.batch_size, num_buckets=cfg.num_buckets)
        return DataLoader(
            ds,
            batch_sampler=sampler,
            num_workers=cfg.num_workers,
            collate_fn=collate_fn,
            pin_memory=(cfg.device.type == 'cuda'),
        )

    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        collate_fn=collate_fn,
        pin_memory=(cfg.device.type == 'cuda'),
    )


def _move_to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def _bf16_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == 'cuda' and _cuda_bf16_supported():
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()


def _adaptive_beam_size(attention_mask: torch.Tensor | None) -> int:
    if not cfg.use_adaptive_beams or attention_mask is None:
        return cfg.num_beams
    med = float(attention_mask.sum(dim=1).float().median().item())
    short = max(cfg.num_beam_cands, cfg.num_beams // 2)
    return short if med < 100 else cfg.num_beams


def postprocess_batch(out_texts: list[str], tasks: list[str]) -> list[str]:
    if len(out_texts) != len(tasks):
        raise ValueError(f"len(out_texts)={len(out_texts)} != len(tasks)={len(tasks)}")
    out: list[str] = ["" for _ in range(len(out_texts))]

    idx_akk2en = [i for i, t in enumerate(tasks) if t == "akk2en"]
    idx_en2akk = [i for i, t in enumerate(tasks) if t == "en2akk"]

    if idx_akk2en:
        texts = [out_texts[i] for i in idx_akk2en]
        pp = _postprocessor.postprocess_batch(texts)
        for i, t in zip(idx_akk2en, pp):
            out[i] = t
    if idx_en2akk:
        texts = [out_texts[i] for i in idx_en2akk]
        pp = _preprocessor.preprocess_batch(texts)
        for i, t in zip(idx_en2akk, pp):
            out[i] = t

    unknown = sorted(set(tasks) - {"akk2en", "en2akk"})
    if unknown:
        raise ValueError(f"Unknown task(s): {unknown}")
    return out


def _is_oom(e: Exception) -> bool:
    msg = str(e).lower()
    return ('out of memory' in msg) or ('cuda error: out of memory' in msg)


def generate_grouped(
    model,
    tokenizer,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor | None,
    *,
    num_return_sequences: int,
    gen_kwargs: dict,
) -> list[list[str]]:
    bs = int(input_ids.size(0))
    try:
        with _bf16_ctx(cfg.device, cfg.use_bf16_amp):
            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=cfg.max_new_tokens,
                **gen_kwargs,
            )
        texts = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        return [
            texts[i * num_return_sequences: (i + 1) * num_return_sequences]
            for i in range(bs)
        ]
    except RuntimeError as e:
        if _is_oom(e):
            print(f'[OOM] generate: bs={bs} num_return={num_return_sequences} gen_kwargs={gen_kwargs}')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if bs == 1:
                raise
            mid = bs // 2
            left = generate_grouped(
                model, tokenizer, input_ids[:mid], attention_mask[:mid] if attention_mask is not None else None,
                num_return_sequences=num_return_sequences, gen_kwargs=gen_kwargs,
            )
            right = generate_grouped(
                model, tokenizer, input_ids[mid:], attention_mask[mid:] if attention_mask is not None else None,
                num_return_sequences=num_return_sequences, gen_kwargs=gen_kwargs,
            )
            return left + right
        raise


# --- MBR selection (chrF++相当) ---
from sacrebleu.metrics import CHRF
_chrf = CHRF(word_order=2)


def _sim(a: str, b: str) -> float:
    return float(_chrf.sentence_score(a, [b]).score) / 100.0


def select_mbr(candidates: list[str]) -> str:
    uniq: list[str] = []
    seen = set()
    for c in candidates:
        c = (c or '').strip()
        if not c:
            continue
        if c in seen:
            continue
        seen.add(c)
        uniq.append(c)
    if not uniq:
        return '<gap>'
    if len(uniq) == 1:
        return uniq[0]

    n = len(uniq)
    sims = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            s = _sim(uniq[i], uniq[j])
            sims[i][j] = s
            sims[j][i] = s

    avg = []
    for i in range(n):
        avg.append((sum(sims[i]) - sims[i][i]) / (n - 1))

    best_i = int(np.argmax(avg))
    return uniq[best_i]


model_specs = [
    ('byt5-large', cfg.byt5_large_dir),
    ('byt5-base', cfg.byt5_base_dir),
    ('byt5-small', cfg.byt5_small_dir),
]

all_candidates: list[list[str]] = [[] for _ in range(len(inputs))]

with torch.inference_mode():
    for model_name, model_dir_raw in model_specs:
        model_dir = resolve_model_dir(model_dir_raw)
        print('---')
        print('Model:', model_name, model_dir)

        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, attn_implementation='eager')
        model.to(cfg.device)
        model.eval()

        if cfg.device.type == 'cuda':
            try:
                torch.set_float32_matmul_precision('high')
            except Exception:
                pass

        if cfg.use_better_transformer and cfg.device.type == 'cuda':
            try:
                from optimum.bettertransformer import BetterTransformer
                model = BetterTransformer.transform(model)
                print(f'BetterTransformer applied: {model_name}')
            except Exception as e:
                print(f'BetterTransformer skipped: {model_name} ({e})')

        dl = make_dataloader(tokenizer, inputs, tasks)

        for batch in tqdm(dl, desc=f'Generating ({model_name})'):
            batch = _move_to_device(batch, cfg.device)
            sample_indices = batch['indices']
            b_tasks = batch['tasks']
            input_ids = batch['input_ids']
            attention_mask = batch.get('attention_mask')

            beam_size = _adaptive_beam_size(attention_mask)

            beam_grouped = generate_grouped(
                model,
                tokenizer,
                input_ids,
                attention_mask,
                num_return_sequences=cfg.num_beam_cands,
                gen_kwargs={
                    'do_sample': False,
                    'num_beams': beam_size,
                    'num_return_sequences': cfg.num_beam_cands,
                    'early_stopping': cfg.early_stopping,
                    'length_penalty': cfg.length_penalty,
                    'repetition_penalty': cfg.repetition_penalty,
                },
            )

            sample_grouped = generate_grouped(
                model,
                tokenizer,
                input_ids,
                attention_mask,
                num_return_sequences=cfg.num_sample_cands,
                gen_kwargs={
                    'do_sample': True,
                    'top_p': cfg.sample_top_p,
                    'temperature': cfg.sample_temperature,
                    'num_beams': 1,
                    'num_return_sequences': cfg.num_sample_cands,
                    'repetition_penalty': cfg.repetition_penalty,
                },
            )

            bs = len(b_tasks)
            for i in range(bs):
                sample_index = sample_indices[i]
                all_candidates[sample_index].extend(beam_grouped[i])
                all_candidates[sample_index].extend(sample_grouped[i])

        try:
            from optimum.bettertransformer import BetterTransformer
            model = BetterTransformer.reverse(model)
        except Exception:
            pass

        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


preds = []
for i, cands in enumerate(tqdm(all_candidates, desc='MBR selecting')):
    task_list = [tasks[i]] * len(cands)
    cands_pp = postprocess_batch(cands, task_list) if cands else []
    preds.append(select_mbr(cands_pp))

# Safety fallback
preds = [('<gap>' if (p is None or str(p).strip() == '') else str(p)) for p in preds]

print('Preds:', len(preds))
print('Example:', preds[0][:200] if preds else '<empty>')


---
Model: byt5-large /kaggle/input/datasets/tatsuokoshida/dpc-byt5-large-v002-6-15epochs


2026-03-22 04:20:55.735726: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774153255.935644      36 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774153255.988889      36 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774153256.436729      36 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774153256.436767      36 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774153256.436770      36 computation_placer.cc:177] computation placer alr

BetterTransformer skipped: byt5-large (No module named 'optimum.bettertransformer')


Generating (byt5-large):   0%|          | 0/4 [00:00<?, ?it/s]

---
Model: byt5-base /kaggle/input/datasets/tatsuokoshida/dpc-byt5-base-v002-6-15epochs
BetterTransformer skipped: byt5-base (No module named 'optimum.bettertransformer')


Generating (byt5-base):   0%|          | 0/4 [00:00<?, ?it/s]

---
Model: byt5-small /kaggle/input/notebooks/tatsuokoshida/2-8-dpc-starter-train-v1/byt5-akkadian-model/fulltrain_byt5-small_multitask
BetterTransformer skipped: byt5-small (No module named 'optimum.bettertransformer')


Generating (byt5-small):   0%|          | 0/4 [00:00<?, ?it/s]

MBR selecting:   0%|          | 0/4 [00:00<?, ?it/s]

Preds: 4
Example: Thus Kanesh, say to the <gap> of our messenger, every single and the wabarrātum: The tablet has come to the City.


In [6]:
# DIFF (2026-03-21): scoring error 対策（空文字→<gap> + validate_submission）は `[4-7-2]` を踏襲
sub_df = pd.DataFrame({"id": ids, "translation": preds})
sub_df["translation"] = sub_df["translation"].fillna("").astype(str)
sub_df.loc[sub_df["translation"].str.strip() == "", "translation"] = "<gap>"

def validate_submission(df: pd.DataFrame, expected_rows: int):
    # Kaggle の format error を避けるため、提出直前に最低限の形を検査する。
    expected_columns = ["id", "translation"]
    if list(df.columns) != expected_columns:
        raise ValueError(f"Submission columns must be {expected_columns}. got={list(df.columns)}")
    if len(df) != expected_rows:
        raise ValueError(f"Submission rows mismatch. expected={expected_rows}, got={len(df)}")
    if df["id"].isna().any():
        raise ValueError("Submission contains missing id values.")
    if df["id"].duplicated().any():
        dup_ids = df.loc[df["id"].duplicated(), "id"].astype(str).head(5).tolist()
        raise ValueError(f"Submission contains duplicated id values. examples={dup_ids}")
    empty_mask = df["translation"].fillna("").astype(str).str.strip() == ""
    if empty_mask.any():
        bad_ids = df.loc[empty_mask, "id"].astype(str).head(5).tolist()
        raise ValueError(f"Submission contains empty translations. examples={bad_ids}")

validate_submission(sub_df, expected_rows=len(ids))

sub_path = Path(cfg.output_dir) / "submission.csv"
sub_df.to_csv(sub_path, index=False)

print("Submission preview:")
display(sub_df.head())
print("Saved to:", str(sub_path))


Submission preview:


,id,translation
0,0,"Thus Kanesh, say to the <gap> of our messenger..."
1,1,In the tablet that was sent to the City assemb...
2,2,"As you have heard our letter, either he has gi..."
3,3,I have sent our tablets to every single and th...


Saved to: /kaggle/working/submission.csv
